# 01 - Data Ingestion

**Objective:** Load the Breast Cancer Wisconsin Diagnostic (WDBC) dataset, clean it, encode the target, and save processed output.

**Dataset:** Breast Cancer Wisconsin Diagnostic (UCI)

**Steps:**
1. Load raw data from `data/raw/wdbc.data`
2. Encode target variable (M→1, B→0)
3. Check data quality (missing values, dtypes)
4. Save processed data


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully")


### Dataset Attributes

The WDBC dataset contains 30 real-valued features computed from digitized images of fine needle aspirate (FNA) of breast masses. Each feature describes characteristics of the cell nuclei present in the image.

**Measurements (10):**

- **Radius:** Mean distance from center to points on the perimeter
- **Texture:** Standard deviation of gray-scale values
- **Perimeter:** Total perimeter of the nucleus
- **Area:** Total area of the nucleus
- **Smoothness:** Local variation in radius lengths
- **Compactness:** Perimeter² / Area − 1.0
- **Concavity:** Severity of concave portions of the contour
- **Concave Points:** Number of concave portions of the contour
- **Symmetry:** Symmetry of the nucleus
- **Fractal Dimension:** "Coastline approximation" − 1

**Statistics (3 per measurement):**

- **Mean:** Average value across all cells in the sample
- **SE:** Standard error across cells
- **Worst:** Mean of the three largest values (most abnormal)

**Target:**

- **Diagnosis:** `M` = Malignant, `B` = Benign — binary classification (what we predict)
- **ID:** Patient identifier — not useful for modeling


In [ ]:
# TODO: Load the raw CSV data into a DataFrame
# The WDBC dataset has no header row, so we pass column names manually.
# Use pd.read_csv() with header=None and names=col_names.

RAW_DIR = Path("../data/raw")

col_names = ["ID", "Diagnosis"]
attributes_cont = [
    "Radius", "Texture", "Perimeter", "Area", "Smoothness",
    "Compactness", "Concavity", "ConcavePoints", "Symmetry", "FractalDimension",
]
for stat in ["Mean", "SE", "Worst"]:
    for feat in attributes_cont:
        col_names.append(f"{feat}_{stat}")

# df = pd.read_csv(RAW_DIR / "wdbc.data", header=None, names=col_names)

# TODO: After loading, inspect the data to make sure it looks right
# Use .shape to confirm we have 569 rows × 32 columns (ID + Diagnosis + 30 features),
# .head() to preview the first few rows, and .info() to check data types.
# The Diagnosis column contains "M" and "B" (object type), the 30 feature columns
# are already numeric (float64), and the ID column is a large integer.

# print(f"Shape: {df.shape}")
# print(df.head())
# df.info()


In [ ]:
# TODO: Convert the Diagnosis column to binary (1 for Malignant "M", 0 for Benign "B") using .apply() or .map().
# This will make it easier to work with the target variable in machine learning models.


# TODO: Use .describe() to get summary statistics for the numeric columns. This will help you understand the distribution of the data and identify any potential outliers.
# print(df.describe())


In [ ]:
# TODO: Check for missing values in each column
# Use df.isnull().sum() to get a count of NaN entries per column.
# This tells you which columns have gaps and how many rows need attention.

# TODO: Handle missing values if any are found
# Two common strategies:
#   - df.dropna() — removes any row with a missing value (simple, but you lose data)
#   - df.fillna(value) — replaces missing entries with something like the column mean or median
# Choose the approach that makes sense for your data.

# TODO: Reset the DataFrame index after dropping rows (if you used dropna)
# df.reset_index(drop=True, inplace=True)



In [ ]:
# TODO: Drop any irrelevant columns that won't be used for modeling (if any)
# For example, if there are columns that are just identifiers or have too many unique values, they might not be useful for prediction and could be dropped to simplify the dataset.
# df.drop(columns=["Irrelevant_Column"], inplace=True) # Replace with actual column names if needed

In [ ]:
# TODO: Change column names to be more Pythonic (optional but recommended) - data specific. 
# For example, if you have a column named "Temperature(°C)", you might want to rename it to "temperature_c" for easier access in code.
# df.rename(columns={"Temperature(°C)": "temperature_c"}, inplace=True)

In [ ]:
# TODO: Find duplicate rows in the DataFrame
# Use df.duplicated() to identify duplicate rows. 
# This returns a boolean Series where True indicates a duplicate row (except for the first occurrence).
# You can sum this Series to get the total count of duplicates.


# TODO: Duplicates can be searched for across all columns or a subset of columns.
# e.g. df.duplicated(subset=["Date", "Hour"])

# TODO: Remove duplicate rows if any are found
# Use df.drop_duplicates() to remove duplicate rows.
# param: keep="first": it keeps the first occurrence and drops subsequent duplicates. alternatives are "last" (keeps the last occurrence) or False (drops all duplicates).
# You can specify subset=["Date", "Hour"] to drop duplicates based on just those columns.


# TODO: Alternatively, you can use df.duplicated() to filter the DataFrame and inspect duplicates before dropping them.

In [ ]:
# TODO: Save the processed data to data/processed/clean_data.csv
# Use df.to_csv() with index=False so we don't write the DataFrame index as a column.

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# df.to_csv(PROCESSED_DIR / "clean_data.csv", index=False)
# print("Clean data saved successfully")


In [ ]:
# TODO: Verify the saved file by loading it back into a new DataFrame
# This step confirms the file wasn't corrupted during the write process.
# Compare the shape of the loaded data to the original shape.

# df_check = pd.read_csv(PROCESSED_DIR / "clean_data.csv")
# print(f"Loaded data shape: {df_check.shape}")
# assert df_check.shape == df.shape, "Shape mismatch after save-load cycle"
# print("Verification passed: data round-trips correctly")


## 2. Relational Data — Joining Tables

In real-world projects, data is rarely handed to you in a single tidy CSV. 
It is often spread across multiple tables in a database — a **fact** table 
and several **dimension** tables, linked by key columns.

To build your feature matrix you need to **join** these tables back together.

In this section you will work with a relational version of the WDBC data,
split into three CSV files that simulate a real pathology database:

| File | Content | Key column |
|------|---------|------------|
| `patient_diagnosis.csv` | Patient ID + diagnosis label | `sample_id` |
| `mean_measurements.csv` | Mean + SE cell measurements (20 cols) | `mean_id` |
| `worst_measurements.csv` | Worst cell measurements (10 cols) | `worst_id` |

Some IDs appear in only some tables — exactly like a real database where lab 
results are missing for certain samples or extra records exist. 
Your job is to re-assemble the full dataset.

In [ ]:
# Load the three relational tables
RELATIONAL_DIR = Path("../data/raw/relational")

diagnosis = pd.read_csv(RELATIONAL_DIR / "patient_diagnosis.csv")
mean_tbl = pd.read_csv(RELATIONAL_DIR / "mean_measurements.csv")
worst_tbl = pd.read_csv(RELATIONAL_DIR / "worst_measurements.csv")

# TODO: Inspect each table
# Check the shape and first few rows of each DataFrame.
# How many IDs does each table have? Are the ID ranges the same?

# print(f"Diagnosis: {diagnosis.shape}")
# print(f"Mean: {mean_tbl.shape}")
# print(f"Worst: {worst_tbl.shape}")
# print(diagnosis.head())
# print(mean_tbl.columns.tolist()[:5])


In [ ]:
# TODO: Merge diagnosis with mean measurements
# Use pd.merge() to join diagnosis with mean_tbl on sample_id == mean_id.
# Start with a LEFT JOIN so you keep all 569 patients.
# Then count how many rows have NaN — those are patients without lab results.

# merged = pd.merge(diagnosis, mean_tbl, left_on="sample_id", right_on="mean_id", how="left")

# TODO: Now try an INNER JOIN. How many rows do you lose? Why?


In [ ]:
# TODO: Merge in the worst_measurements table too
# Chain a second merge to bring in the Worst cell measurements.

# TODO: Check how many rows have missing Worst measurements


In [ ]:
# TODO: Compare to the single-table original
# The original wdbc.data has 569 rows with all 30 features present.
# After splitting and rejoining, how many rows have all measurements?


# HINT: The difference between 569 and len(complete) tells you how many
# records are missing from each lab table.


### Exercises

1. **Explore encoding options**: The Diagnosis column uses M=1, B=0. What happens if you reverse the mapping (M=0, B=1)? Does it change the model? Why or why not?
2. **Experiment with missing value handling**: Intentionally set a few values in the first feature column to NaN with `df.iloc[:10, 0] = np.nan`, then try both `dropna()` and `fillna(df.mean())` and compare the resulting shapes.
3. **Save in Parquet format**: Use `df.to_parquet()` instead of CSV and compare file sizes. Parquet is often faster and more space-efficient for numeric data.
4. **Feature engineering idea**: Create a new feature `size_ratio = Radius_Mean / Area_Mean`. Does this ratio capture something different from either raw measurement?
5. **What would break?**: If the raw data file had an extra column inserted between ID and Diagnosis, which part of this notebook would fail first? What if the first row contained column names instead of data?
6. **Which JOIN?**: You used LEFT JOINs to assemble the relational tables. Try replacing them with INNER JOINs. How many rows remain after the second merge? Why?
7. **Fake data detective**: The mean and worst measurement tables contain artificially generated rows (IDs > 569). Can you identify them? What heuristic did you use?
